# Ayurveda Shloka Benchmark — model selection runnerScores a candidate base model on **100 questions** drawn from the AyurShloka-QA set(25 each: assertion-reasoning, MCQ, fill-in-the-blanks, match-the-column), so you canpick which model to trim and fine-tune on the remaining ~21k questions.All four task types are 4-option items, so each one reports **both**:* **classification** — accuracy, precision/recall/F1 (micro / macro / weighted), confusion matrix* **generation** — exact match, token P/R/F1, ROUGE-1/2/L, BLEU, METEOR, BERTScoreplus latency, tokens/sec and **format compliance** (how often the model emitted aparseable answer at all — a low value means the score reflects instruction-following,not Ayurvedic knowledge).---## 🔧 Cells you need to edit| Cell | What to change ||---|---|| **Section 6 — CONFIG** | `MODEL_NAME` — the only required edit || **Section 12 — Full run** | `BATCH_SIZE`, `LOAD_IN_4BIT`, `USE_CHAT_TEMPLATE` per model || **Section 16 — Recommend** | `WEIGHTS`, if some task types matter more than others |**Nothing to attach.** The scoring engine (Section 4) and all 100 questions(Section 5) are embedded in this notebook. Import it, set `MODEL_NAME`, Run All.Everything else runs as-is.**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings → Internet → **On**(needed for model downloads, BERTScore and the NLTK corpora).

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch rouge-score nltk python-dotenv tqdm sacremoses \    scikit-learn bitsandbytes accelerate bert-score!pip install -q -U transformers# METEOR needs the wordnet corpus; without it METEOR silently reports 0.0 for every row.import nltkfor pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:    nltk.download(pkg, quiet=True)print("deps + nltk corpora ready")

## 2. Check GPU

In [ ]:
import torchprint("CUDA available:", torch.cuda.is_available())print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")if torch.cuda.is_available():    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 3. HF token — needed for gated repos (Llama, Gemma)**Add-ons → Secrets → add `HF_TOKEN` → then turn the toggle ON for this notebook.**A saved secret is not visible to a notebook until you enable it there, which is thestep people miss.The token needs *"Read access to contents of all public gated repos you can access"*,and you must separately accept each model's licence on its HuggingFace page.

In [ ]:
import ostry:    from kaggle_secrets import UserSecretsClient    _tokv = UserSecretsClient().get_secret("HF_TOKEN")    os.environ["HF_TOKEN"] = _tokv    os.environ["HUGGING_FACE_HUB_TOKEN"] = _tokv    print("HF_TOKEN loaded — length %d, starts %s..." % (len(_tokv), _tokv[:7]))except Exception as e:    print("No HF_TOKEN secret:", e)    print("Public models still work. Gated repos (meta-llama/*, google/gemma*) will 403.")

## 4. Write the benchmark engineThe full scoring engine is bundled inside this notebook. This cell writes it out as`ayur_benchmark.py` in the working directory, and later cells import it from there — soyou can edit it in place and just re-run the import cell, no kernel restart needed.

In [ ]:
%%writefile ayur_benchmark.py#!/usr/bin/env python3# -*- coding: utf-8 -*-"""Ayurveda Shloka Benchmark — scoring engine==========================================Adapted from the Cardiac-LLM Indic benchmark for the AyurShloka-QA question set.Task types: ``assertion``, ``mcq``, ``fill_blanks``, ``match_column``.All four are 4-option items, so every type gets BOTH:  * classification metrics — accuracy, precision/recall/F1 (micro / macro /    weighted) and a confusion matrix over the predicted option letter;  * generation metrics — exact match, token P/R/F1, ROUGE-1/2/L, BLEU, METEOR    and BERTScore against the full text of the correct option;plus latency and tokens/sec.Import-safe: ``import ayur_benchmark`` performs no downloads, no NLTK fetches andno model loading. Heavy dependencies are imported lazily on first use.Typical use::    import ayur_benchmark as ab    ab.run_benchmark("Qwen/Qwen2.5-3B-Instruct", data_dir="data", results_dir="results")    ab.leaderboard("results")"""from __future__ import annotationsimport csvimport jsonimport osimport reimport stringimport timefrom collections import Counterfrom pathlib import Pathfrom typing import Dict, List, Optional, Sequence__all__ = [    "TASK_TYPES", "FILENAME_MAP", "run_benchmark", "compare_models", "free_vram",    "check_access", "preflight", "hf_token", "GatedModelError", "UNGATED_MIRROR",    "validate_data", "leaderboard", "pivot_view", "recommend",    "load_jsonl", "score_records", "resolve_letter",]# --------------------------------------------------------------------------# Configuration# --------------------------------------------------------------------------TASK_TYPES: List[str] = ["assertion", "mcq", "fill_blanks", "match_column"]#: task type -> expected file name inside ``data_dir``FILENAME_MAP: Dict[str, str] = {    "assertion":    "ayur_assertion_25.jsonl",    "mcq":          "ayur_mcq_25.jsonl",    "fill_blanks":  "ayur_fill_blanks_25.jsonl",    "match_column": "ayur_match_column_25.jsonl",}MAX_PARAMS_DEFAULT = 8_000_000_000          # hard cap: 8B paramsBERTSCORE_MODEL = "bert-base-multilingual-cased"   # multilingual: handles DevanagariCHOICE_LETTERS = ["A", "B", "C", "D"]NO_ANSWER = "NONE"                          # model produced no parseable choice#: per-type instruction appended to the prompt so the model knows the answer formatANSWER_DIRECTIVE: Dict[str, str] = {    "assertion": ("You are an expert in classical Ayurveda. Choose the single correct "                  "option. Reply with ONLY the option letter (A, B, C or D)."),    "mcq": ("You are an expert in classical Ayurveda. Choose the single correct option. "            "Reply with ONLY the option letter (A, B, C or D)."),    "fill_blanks": ("You are an expert in classical Ayurveda. Choose the option that "                    "correctly fills the blank. Reply with ONLY the option letter "                    "(A, B, C or D)."),    "match_column": ("You are an expert in classical Ayurveda. Choose the option giving "                     "the correct set of matches. Reply with ONLY the option letter "                     "(A, B, C or D)."),}_DEPS: dict = {}#: models that are gated on HuggingFace and need an approved licence + token,#: mapped to an ungated mirror that is weight-identical where one existsUNGATED_MIRROR = {    "meta-llama/Llama-3.2-3B-Instruct": "unsloth/Llama-3.2-3B-Instruct",    "meta-llama/Llama-3.2-1B-Instruct": "unsloth/Llama-3.2-1B-Instruct",    "meta-llama/Llama-3.1-8B-Instruct": "unsloth/Meta-Llama-3.1-8B-Instruct",    "google/gemma-2-2b-it": "unsloth/gemma-2-2b-it",    "google/gemma-2-9b-it": "unsloth/gemma-2-9b-it",}def hf_token() -> Optional[str]:    """The HuggingFace token from the environment, if one is set."""    for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"):        v = os.environ.get(k)        if v and v.strip():            return v.strip()    return Nonedef _is_gated_error(e) -> bool:    t = "%s %s" % (type(e).__name__, e)    return ("GatedRepo" in t or "gated repo" in t.lower()            or "restricted and you are not in the authorized list" in t)class GatedModelError(RuntimeError):    """Raised with actionable instructions instead of a raw 403 traceback."""def _gated_message(model_name: str) -> str:    tok = hf_token()    mirror = UNGATED_MIRROR.get(model_name)    lines = [        "GATED REPO: %s cannot be downloaded with the current credentials." % model_name,        "",        "HF_TOKEN is %s." % ("set" if tok else "NOT set in this kernel"),        "",        "To use this exact repo, all three must be true:",        "  1. Accept the licence at https://huggingface.co/%s" % model_name,        "     (approval is usually quick, but it is a manual gate on Meta's side)",        "  2. Create a token at https://huggingface.co/settings/tokens with",        "     'Read access to contents of all public gated repos you can access'",        "  3. On Kaggle: Add-ons -> Secrets -> add it as HF_TOKEN, then make sure the",        "     toggle next to it is ON *for this notebook* - a saved secret is not",        "     attached to a notebook until you enable it there.",    ]    if mirror:        lines += ["", "Or skip the gate entirely - this ungated mirror carries the same weights:",                  "     MODEL_NAME = \"%s\"" % mirror]    return "\n".join(lines)def check_access(model_name: str) -> dict:    """Can this repo actually be downloaded? Cheap - fetches the config only."""    try:        from transformers import AutoConfig        AutoConfig.from_pretrained(model_name, token=hf_token(),                                   trust_remote_code=False)        return {"model": model_name, "ok": True, "reason": ""}    except Exception as e:        if _is_gated_error(e):            return {"model": model_name, "ok": False, "reason": "gated",                    "mirror": UNGATED_MIRROR.get(model_name, "")}        # a config that needs remote code is still accessible        if "trust_remote_code" in str(e) or "custom" in str(e).lower():            return {"model": model_name, "ok": True, "reason": "needs trust_remote_code"}        return {"model": model_name, "ok": False,                "reason": "%s: %s" % (type(e).__name__, str(e)[:120])}def preflight(model_names: Sequence[str]) -> List[str]:    """Report which models are downloadable; return the usable list."""    print("%-42s %s" % ("model", "status"))    print("-" * 78)    usable = []    for m in model_names:        r = check_access(m)        if r["ok"]:            usable.append(m)            print("%-42s OK%s" % (m, "  (" + r["reason"] + ")" if r["reason"] else ""))        elif r["reason"] == "gated":            alt = r.get("mirror")            print("%-42s GATED%s" % (m, "  -> try " + alt if alt else ""))            if alt:                usable.append(alt)        else:            print("%-42s FAIL  %s" % (m, r["reason"]))    print("\nusable: %d of %d" % (len(usable), len(model_names)))    if not hf_token():        print("note: HF_TOKEN is not set - gated repos will fail")    return usable# --------------------------------------------------------------------------# Lazy dependency loading (keeps `import ayur_benchmark` instant + side-effect free)# --------------------------------------------------------------------------def _ensure_scoring_deps():    """Import + prepare the metric libraries. Safe to call repeatedly."""    if _DEPS.get("scoring"):        return _DEPS    from rouge_score import rouge_scorer    import nltk    from nltk.translate.bleu_score import SmoothingFunction    from sklearn.metrics import (        f1_score, precision_score, recall_score, accuracy_score, confusion_matrix,    )    for pkg, path in [("punkt", "tokenizers/punkt"), ("punkt_tab", "tokenizers/punkt_tab"),                      ("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")]:        try:            nltk.data.find(path)        except LookupError:            try:                nltk.download(pkg, quiet=True)            except Exception:                pass    _DEPS.update(        scoring=True,        rouge=rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True),        smoothing=SmoothingFunction().method1,        nltk=nltk,        f1_score=f1_score, precision_score=precision_score, recall_score=recall_score,        accuracy_score=accuracy_score, confusion_matrix=confusion_matrix,    )    return _DEPSdef _ensure_torch():    if "torch" not in _DEPS:        import torch        _DEPS["torch"] = torch    return _DEPS["torch"]# --------------------------------------------------------------------------# Text utilities# --------------------------------------------------------------------------def normalize_text(s: str) -> str:    s = (s or "").lower()    s = "".join(ch for ch in s if ch not in set(string.punctuation))    s = re.sub(r"\b(a|an|the)\b", " ", s)    return re.sub(r"\s+", " ", s).strip()def exact_match(pred: str, gold: str) -> int:    return int(normalize_text(pred) == normalize_text(gold))def token_prf(pred: str, gold: str):    """SQuAD-style token overlap -> precision, recall, f1, tp, fp, fn."""    p_toks, g_toks = normalize_text(pred).split(), normalize_text(gold).split()    common = Counter(p_toks) & Counter(g_toks)    tp = sum(common.values())    fp, fn = len(p_toks) - tp, len(g_toks) - tp    if tp == 0:        return 0.0, 0.0, 0.0, tp, fp, fn    precision = tp / (tp + fp) if (tp + fp) else 0.0    recall = tp / (tp + fn) if (tp + fn) else 0.0    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0    return precision, recall, f1, tp, fp, fndef _token_f1(a: str, b: str) -> float:    return token_prf(a, b)[2]# ---- answer extraction ----------------------------------------------------# A leading option letter must be followed by a delimiter, a line break, or the end of# the string. Notably this accepts "A\n\nExplanation: ..." - the single most common# instruct-model response shape - while still refusing "A patient presents with ...",# where "A" is the English article._LEADING_LETTER = re.compile(    r"^\s*[\*_`\"'(\[]*\s*([A-D])\s*(?:[\)\].:,\-—*_`]|\r?\n|$)")_LEADING_LETTER_VERB = re.compile(    r"^\s*\(?([A-D])\)?\s+(?=is\b|would\b|seems\b|appears\b|correct\b|option\b)",    re.IGNORECASE)_BARE_LETTER = re.compile(r"^\s*[\*_`\"'(\[]*\s*([A-D])\s*[\)\]\*_`\"'.]*\s*$")_STATED_ANSWER = re.compile(    r"(?:final\s+)?(?:answer|option|choice|correct(?:\s+option)?)\s*(?:is|:|-|=)?\s*"    r"[\*_`(\[]*\s*([A-D])\b", re.IGNORECASE)_BOLD_LETTER = re.compile(r"\*\*\s*\(?([A-D])\)?\s*\*\*")_MATCH_KEY = re.compile(r"\b(i{1,3}|iv)\s*[-–—:]\s*([abcd])\b", re.IGNORECASE)def _match_permutation(text: str) -> Optional[str]:    """Pull an 'i-c, ii-a, iii-b, iv-d' style mapping out of free text."""    pairs = _MATCH_KEY.findall(text or "")    seen = {}    for roman, letter in pairs:        roman = roman.lower()        if roman in ("i", "ii", "iii", "iv") and roman not in seen:            seen[roman] = letter.lower()    if len(seen) == 4:        return ", ".join("%s-%s" % (r, seen[r]) for r in ("i", "ii", "iii", "iv"))    return Nonedef resolve_letter(pred: str, options: Optional[Dict[str, str]] = None) -> str:    """Map a raw generation onto an option letter.    Tries, in order: an explicit stated answer, a leading/bare/bold letter, a    verbatim match against an option's text, a match-the-column permutation, and    finally best token-overlap against the option texts. Returns ``NONE`` when the    model produced nothing usable — which is itself worth seeing in the confusion    matrix, since it separates "wrong" from "did not follow the format".    """    text = (pred or "").strip()    if not text:        return NO_ANSWER    for rx in (_STATED_ANSWER, _BOLD_LETTER, _BARE_LETTER,               _LEADING_LETTER, _LEADING_LETTER_VERB):        m = rx.match(text) if rx in (_BARE_LETTER, _LEADING_LETTER,                                     _LEADING_LETTER_VERB) else rx.search(text)        if m:            return m.group(1).upper()    if options:        norm_pred = normalize_text(text)        # verbatim containment — longest option first to avoid substring collisions        for letter, opt in sorted(options.items(), key=lambda kv: -len(kv[1] or "")):            n_opt = normalize_text(opt)            if len(n_opt) >= 8 and n_opt in norm_pred:                return letter.upper()        # match-the-column permutation stated in prose        perm = _match_permutation(text)        if perm:            for letter, opt in options.items():                if _match_permutation(opt) == perm:                    return letter.upper()        # fuzzy fallback        scored = [(_token_f1(text, opt or ""), letter) for letter, opt in options.items()]        best_score, best_letter = max(scored)        runner_up = sorted((s for s, _ in scored), reverse=True)[1] if len(scored) > 1 else 0.0        if best_score >= 0.55 and best_score - runner_up >= 0.10:            return best_letter.upper()    return NO_ANSWER# ---- generation metrics ---------------------------------------------------def rouge_scores(pred: str, gold: str):    d = _ensure_scoring_deps()    s = d["rouge"].score(gold, pred)    return s["rouge1"].fmeasure, s["rouge2"].fmeasure, s["rougeL"].fmeasuredef bleu_score(pred: str, gold: str) -> float:    d = _ensure_scoring_deps()    from nltk.translate.bleu_score import sentence_bleu    ref, hyp = normalize_text(gold).split(), normalize_text(pred).split()    if not hyp or not ref:        return 0.0    try:        return sentence_bleu([ref], hyp, smoothing_function=d["smoothing"])    except Exception:        return 0.0def meteor(pred: str, gold: str) -> float:    _ensure_scoring_deps()    from nltk.translate.meteor_score import meteor_score    ref, hyp = normalize_text(gold).split(), normalize_text(pred).split()    if not hyp or not ref:        return 0.0    try:        return meteor_score([ref], hyp)    except Exception as e:        # Silent zeros here would look like a model failure rather than a missing        # corpus, so say it once, loudly.        if not _DEPS.get("meteor_warned"):            _DEPS["meteor_warned"] = True            print("  [WARN] METEOR unavailable (%s: %s) — the nltk 'wordnet' corpus is "                  "missing, so METEOR will report 0.0 for every row. Fix with "                  "`import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')` "                  "(needs Internet ON). All other metrics are unaffected."                  % (type(e).__name__, e))        return 0.0def bertscore_batch(preds: Sequence[str], golds: Sequence[str], device: str):    """Batched BERTScore across a whole task type — far faster than per-example."""    from bert_score import score as bertscore_score    safe = [(p if (p or "").strip() else " ", g if (g or "").strip() else " ")            for p, g in zip(preds, golds)]    P, R, F1 = bertscore_score(        [p for p, _ in safe], [g for _, g in safe],        model_type=BERTSCORE_MODEL, lang=None, rescale_with_baseline=False,        device=device if "cuda" in str(device) else "cpu", verbose=False,    )    return P.tolist(), R.tolist(), F1.tolist()# --------------------------------------------------------------------------# Data loading / validation# --------------------------------------------------------------------------def load_jsonl(path) -> List[dict]:    rows = []    with open(path, "r", encoding="utf-8") as f:        for line in f:            line = line.strip()            if line:                rows.append(json.loads(line))    return rowsdef _parse_options(row: dict) -> Dict[str, str]:    raw = row.get("input") or ""    if isinstance(raw, dict):        return {k.upper(): v for k, v in raw.items()}    if not str(raw).strip():        return {}    try:        return {k.upper(): v for k, v in json.loads(raw).items()}    except Exception:        return {}def validate_data(data_dir="data", types: Optional[Sequence[str]] = None) -> bool:    """Print a validation report for the eval files. Returns True if all clean."""    data_dir = Path(data_dir)    types = list(types or TASK_TYPES)    print("=" * 72)    print("AYURVEDA BENCHMARK — DATASET VALIDATION")    print("=" * 72)    ok = True    for dtype in types:        path = data_dir / FILENAME_MAP[dtype]        print("\n--- %s ---" % dtype.upper())        if not path.exists():            print("  FILE NOT FOUND: %s" % path)            ok = False            continue        rows = load_jsonl(path)        bad_opts = sum(1 for r in rows if len(_parse_options(r)) != 4)        bad_key = sum(1 for r in rows                      if str(r.get("output", "")).strip().upper() not in CHOICE_LETTERS)        mismatched = 0        for r in rows:            opts = _parse_options(r)            key = str(r.get("output", "")).strip().upper()            if opts and key in opts and r.get("output_text"):                if normalize_text(opts[key]) != normalize_text(r["output_text"]):                    mismatched += 1        empty_instr = sum(1 for r in rows if not str(r.get("instruction", "")).strip())        books = Counter(r.get("source_book", "?") for r in rows)        subs = Counter(r.get("question_subtype", "?") for r in rows)        letters = Counter(str(r.get("output", "")).upper() for r in rows)        print("  Rows                : %d" % len(rows))        print("  Empty instruction   : %d" % empty_instr)        print("  Rows without 4 opts : %d" % bad_opts)        print("  Bad answer key      : %d" % bad_key)        print("  Key/text mismatch   : %d" % mismatched)        print("  Answer letters      : %s" % dict(sorted(letters.items())))        print("  Books               : %d  %s" % (len(books), dict(books.most_common())))        print("  Subtypes            : %s" % dict(subs.most_common()))        if empty_instr or bad_opts or bad_key or mismatched:            ok = False            print("  WARNING: issues above will distort scores")    print("\n%s" % ("DATASET LOOKS CLEAN." if ok else "DATASET HAS ISSUES — fix before trusting scores."))    return ok# --------------------------------------------------------------------------# Model loading + generation# --------------------------------------------------------------------------def load_model(model_name: str, device: str = "cuda", force_size: bool = False,               load_in_4bit: bool = False, max_params: int = MAX_PARAMS_DEFAULT,               trust_remote_code: Optional[bool] = None):    """Load a model, trying several loading strategies before giving up.    ``trust_remote_code`` defaults to None, meaning "prefer the implementation built    into transformers, and only fall back to the repo's own modeling code if that    fails". This matters: several popular repos ship modeling code that has drifted    out of sync with current transformers. Phi-3 is the classic case — its bundled    ``modeling_phi3.py`` reads ``config.rope_scaling["type"]``, a key transformers    renamed to ``rope_type``, so forcing remote code raises ``KeyError: 'type'`` even    though transformers supports Phi-3 natively and loads it fine.    Pass True or False to force the choice.    """    torch = _ensure_torch()    from transformers import AutoTokenizer, AutoModelForCausalLM    tok_trc = True if trust_remote_code is None else trust_remote_code    tk = hf_token()    try:        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=tok_trc, token=tk)    except Exception as e:        if _is_gated_error(e):            raise GatedModelError(_gated_message(model_name)) from None        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=not tok_trc,                                                  token=tk)    if tokenizer.pad_token is None:        tokenizer.pad_token = tokenizer.eos_token    # decoder-only models MUST be left-padded for batched generation, otherwise the    # generated continuation starts after the pad run and comes back empty/garbage    tokenizer.padding_side = "left"    dtype = torch.float16 if "cuda" in device else torch.float32    base = dict(device_map={"": 0} if "cuda" in device else None, token=tk)    if load_in_4bit:        from transformers import BitsAndBytesConfig        base["quantization_config"] = BitsAndBytesConfig(            load_in_4bit=True,            bnb_4bit_compute_dtype=torch.float16,            bnb_4bit_quant_type="nf4",            bnb_4bit_use_double_quant=True,        )    else:        base["dtype"] = dtype    if trust_remote_code is None:        trc_order = [False, True]      # native implementation first    else:        trc_order = [trust_remote_code]    attempts = []    for trc in trc_order:        for extra in ({}, {"attn_implementation": "eager"}, {"use_safetensors": False}):            attempts.append(dict(trust_remote_code=trc, **extra))    last = None    for i, extra in enumerate(attempts):        kw = dict(base)        kw.update(extra)        for dtype_key in ("dtype", "torch_dtype"):            k = dict(kw)            if "dtype" in k and dtype_key == "torch_dtype":                k["torch_dtype"] = k.pop("dtype")            try:                model = AutoModelForCausalLM.from_pretrained(model_name, **k)                if i:                    print("  (loaded with %s)"                          % ", ".join("%s=%s" % (a, b) for a, b in extra.items()))                n_params = sum(p.numel() for p in model.parameters())                print("Model params: %.2fB" % (n_params / 1e9))                if n_params > max_params and not force_size:                    raise RuntimeError(                        "REJECTED: %s has %.2fB params — cap is %.1fB. Pass "                        "force_size=True to override."                        % (model_name, n_params / 1e9, max_params / 1e9))                model.eval()                return tokenizer, model            except RuntimeError:                raise            except TypeError:                continue                      # wrong dtype kwarg for this version            except Exception as e:            # noqa: BLE001 - try the next strategy                if _is_gated_error(e):                    raise GatedModelError(_gated_message(model_name)) from None                last = e                break    raise RuntimeError(        "Could not load %s after %d strategies. Last error: %s: %s\n"        "If this is a KeyError on a config field, the repo's modeling code is out of "        "sync with your transformers version — try trust_remote_code=False, or "        "`pip install -U transformers`."        % (model_name, len(attempts), type(last).__name__, last))def build_prompt(row: dict, dtype: str, tokenizer=None, use_chat_template: bool = True,                 use_context: bool = False) -> str:    """Assemble the prompt. Applies the model's chat template when it has one."""    parts = []    if use_context and str(row.get("context", "")).strip():        parts.append("Context:\n%s" % row["context"])    parts.append(row.get("instruction", ""))    opts = _parse_options(row)    if opts:        parts.append("\n".join("(%s) %s" % (L, opts[L]) for L in CHOICE_LETTERS if L in opts))    body = "\n\n".join(p for p in parts if p)    directive = ANSWER_DIRECTIVE.get(dtype, "Reply with ONLY the option letter.")    if use_chat_template and tokenizer is not None and getattr(tokenizer, "chat_template", None):        messages = [{"role": "user", "content": "%s\n\n%s" % (directive, body)}]        try:            return tokenizer.apply_chat_template(                messages, tokenize=False, add_generation_prompt=True)        except Exception:            pass    return "%s\n\n%s\n\nAnswer:" % (directive, body)def generate_batch(tokenizer, model, device, prompts, max_new_tokens=24):    torch = _ensure_torch()    enc = tokenizer(prompts, return_tensors="pt", padding=True,                    truncation=True, max_length=2048).to(device)    with torch.no_grad():        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,                             pad_token_id=tokenizer.pad_token_id)    texts = []    for i in range(out.shape[0]):        gen_ids = out[i][enc["input_ids"].shape[1]:]        texts.append(tokenizer.decode(gen_ids, skip_special_tokens=True).strip())    return texts# --------------------------------------------------------------------------# Scoring# --------------------------------------------------------------------------def score_records(records: List[dict], device: str = "cpu", skip_bertscore: bool = False):    """Score one task type. Returns (metrics dict, confusion dict)."""    if not records:        return {}, None    d = _ensure_scoring_deps()    n = len(records)    # ---------- classification over the option letter ----------    y_true = [r["gold_letter"] for r in records]    y_pred = [r["pred_letter"] for r in records]    labels = sorted(set(y_true) | set(y_pred))    metrics = {        "n": n,        "accuracy": round(d["accuracy_score"](y_true, y_pred), 4),        "f1_micro": round(d["f1_score"](y_true, y_pred, labels=labels, average="micro", zero_division=0), 4),        "f1_macro": round(d["f1_score"](y_true, y_pred, labels=labels, average="macro", zero_division=0), 4),        "f1_weighted": round(d["f1_score"](y_true, y_pred, labels=labels, average="weighted", zero_division=0), 4),        "precision_micro": round(d["precision_score"](y_true, y_pred, labels=labels, average="micro", zero_division=0), 4),        "precision_macro": round(d["precision_score"](y_true, y_pred, labels=labels, average="macro", zero_division=0), 4),        "recall_micro": round(d["recall_score"](y_true, y_pred, labels=labels, average="micro", zero_division=0), 4),        "recall_macro": round(d["recall_score"](y_true, y_pred, labels=labels, average="macro", zero_division=0), 4),        "format_compliance": round(sum(1 for p in y_pred if p != NO_ANSWER) / n, 4),        "random_baseline": 0.25,    }    cm = d["confusion_matrix"](y_true, y_pred, labels=labels)    confusion = {"labels": labels, "matrix": cm.tolist()}    # ---------- generation metrics against the correct option's text ----------    preds = [r["pred"] for r in records]    golds = [r["gold_text"] for r in records]    ems, f1s, precs, recs, r1s, r2s, rLs, bl, mt = ([] for _ in range(9))    tp_sum = fp_sum = fn_sum = 0    for pred, gold in zip(preds, golds):        ems.append(exact_match(pred, gold))        p, rc, f1, tp, fp, fn = token_prf(pred, gold)        precs.append(p); recs.append(rc); f1s.append(f1)        tp_sum += tp; fp_sum += fp; fn_sum += fn        a, b, c = rouge_scores(pred, gold)        r1s.append(a); r2s.append(b); rLs.append(c)        bl.append(bleu_score(pred, gold))        mt.append(meteor(pred, gold))    micro_p = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) else 0.0    micro_r = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) else 0.0    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0    metrics.update({        "exact_match": round(sum(ems) / n, 4),        "token_precision_macro": round(sum(precs) / n, 4),        "token_recall_macro": round(sum(recs) / n, 4),        "token_f1_macro": round(sum(f1s) / n, 4),        "token_precision_micro": round(micro_p, 4),        "token_recall_micro": round(micro_r, 4),        "token_f1_micro": round(micro_f1, 4),        "rouge1": round(sum(r1s) / n, 4),        "rouge2": round(sum(r2s) / n, 4),        "rougeL": round(sum(rLs) / n, 4),        "bleu": round(sum(bl) / n, 4),        "meteor": round(sum(mt) / n, 4),    })    if skip_bertscore:        metrics.update(bertscore_precision=None, bertscore_recall=None, bertscore_f1=None)    else:        try:            bp, br, bf = bertscore_batch(preds, golds, device)            metrics.update(                bertscore_precision=round(sum(bp) / n, 4),                bertscore_recall=round(sum(br) / n, 4),                bertscore_f1=round(sum(bf) / n, 4),            )        except Exception as e:            print("  BERTScore failed (%s) — reporting 0.0" % e)            metrics.update(bertscore_precision=0.0, bertscore_recall=0.0, bertscore_f1=0.0)    return metrics, confusiondef run_type(dtype, tokenizer, model, device, data_dir, batch_size, limit,             max_new_tokens, use_chat_template, use_context, skip_bertscore):    path = Path(data_dir) / FILENAME_MAP[dtype]    if not path.exists():        print("  (skip) %s not found" % path)        return None, [], None    rows = load_jsonl(path)    if limit:        rows = rows[:limit]    records, total_new_tokens = [], 0    t0 = time.time()    for i in range(0, len(rows), batch_size):        chunk = rows[i:i + batch_size]        prompts = [build_prompt(r, dtype, tokenizer, use_chat_template, use_context)                   for r in chunk]        preds = generate_batch(tokenizer, model, device, prompts, max_new_tokens)        for r, pred in zip(chunk, preds):            total_new_tokens += len(tokenizer.encode(pred))            opts = _parse_options(r)            gold_letter = str(r.get("output", "")).strip().upper()            records.append({                "type": dtype,                "id": r.get("id", ""),                "instruction": r.get("instruction", ""),                "input": r.get("input", ""),                "gold_letter": gold_letter,                "gold_text": r.get("output_text", opts.get(gold_letter, "")),                "pred": pred,                "pred_letter": resolve_letter(pred, opts),                "correct": "",                "source": r.get("source", ""),                "source_book": r.get("source_book", ""),                "difficulty": r.get("difficulty", ""),                "question_subtype": r.get("question_subtype", ""),                "subject_domain": r.get("subject_domain", ""),                "language": r.get("language", "en"),            })    for rec in records:        rec["correct"] = int(rec["pred_letter"] == rec["gold_letter"])    elapsed = time.time() - t0    metrics, confusion = score_records(records, device, skip_bertscore)    metrics["latency_sec"] = round(elapsed, 2)    metrics["tokens_per_sec"] = round(total_new_tokens / elapsed if elapsed > 0 else 0.0, 2)    return metrics, records, confusion# --------------------------------------------------------------------------# Main entry point# --------------------------------------------------------------------------def run_benchmark(model_name: str,                  data_dir: str = "data",                  results_dir: str = "results",                  device: str = "cuda",                  limit: int = 0,                  batch_size: int = 4,                  max_new_tokens: int = 24,                  types: Optional[Sequence[str]] = None,                  force_size: bool = False,                  save_results: bool = True,                  load_in_4bit: bool = False,                  use_chat_template: bool = True,                  use_context: bool = False,                  skip_bertscore: bool = False,                  trust_remote_code: Optional[bool] = None,                  max_params: int = MAX_PARAMS_DEFAULT) -> dict:    """Benchmark one model over the Ayurveda question sets.    Args:        model_name: full HuggingFace repo id, e.g. ``"Qwen/Qwen2.5-3B-Instruct"``.        data_dir: folder holding the four ``ayur_*_25.jsonl`` files.        results_dir: where CSVs and the report are written.        limit: cap rows per type (0 = all). Use with ``save_results=False`` for smoke tests.        max_new_tokens: keep small (16-32); these are letter-answer tasks.        load_in_4bit: quantise so 7-8B models fit on a single T4.        use_chat_template: apply the model's chat template when it has one. Turn OFF            for base (non-instruct) models.        use_context: also feed the source passage. Off by default — with context the            task becomes reading comprehension rather than recall of the texts.        skip_bertscore: skip the BERTScore backbone download (faster smoke tests).        trust_remote_code: None (default) tries the transformers-native implementation            first and falls back to the repo's own modeling code. Set False to force            native — needed for repos whose bundled code has drifted (e.g. Phi-3            raising ``KeyError: 'type'`` on ``rope_scaling``).        save_results: False prints metrics but writes nothing, so quick runs never            pollute leaderboard.csv.    """    types = list(types or TASK_TYPES)    results_dir = Path(results_dir)    print("\nModel      : %s" % model_name)    print("Device     : %s" % device)    print("Data dir   : %s" % data_dir)    print("Types      : %s" % types)    print("Limit      : %s" % (limit or "ALL"))    print("4-bit      : %s" % load_in_4bit)    print("Chat tmpl  : %s" % use_chat_template)    print("Saving     : %s\n" % ("yes" if save_results else "no (sanity-check mode)"))    tokenizer, model = load_model(model_name, device, force_size, load_in_4bit,                                  max_params, trust_remote_code)    all_metrics, all_records, confusions = {}, [], {}    for dtype in types:        print("\n--- Running %s ---" % dtype)        metrics, records, confusion = run_type(            dtype, tokenizer, model, device, data_dir, batch_size, limit,            max_new_tokens, use_chat_template, use_context, skip_bertscore)        if metrics:            all_metrics[dtype] = metrics            all_records.extend(records)            print("  accuracy=%.4f  f1_macro=%.4f  format_compliance=%.2f  tok/s=%.1f"                  % (metrics["accuracy"], metrics["f1_macro"],                     metrics["format_compliance"], metrics["tokens_per_sec"]))        if confusion:            confusions[dtype] = confusion    if all_metrics:        overall = sum(m["accuracy"] * m["n"] for m in all_metrics.values()) / \                  sum(m["n"] for m in all_metrics.values())        print("\nOVERALL ACCURACY: %.4f  (random baseline 0.25)" % overall)    if not save_results:        print("\n(sanity-check mode: nothing written to %s)" % results_dir)        return all_metrics    results_dir.mkdir(parents=True, exist_ok=True)    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)    if all_records:        with open(results_dir / ("benchmark_results_%s.csv" % safe), "w",                  newline="", encoding="utf-8") as f:            w = csv.DictWriter(f, fieldnames=list(all_records[0].keys()))            w.writeheader()            w.writerows(all_records)    for dtype, conf in confusions.items():        p = results_dir / ("confusion_matrix_%s_%s.csv" % (dtype, safe))        with open(p, "w", newline="", encoding="utf-8") as f:            w = csv.writer(f)            w.writerow(["true\\pred"] + conf["labels"])            for label, row in zip(conf["labels"], conf["matrix"]):                w.writerow([label] + row)        print("Wrote confusion matrix: %s" % p)    lb = results_dir / "leaderboard.csv"    rows_out = [{"model": model_name, "type": dt, **m} for dt, m in all_metrics.items()]    if rows_out:        existing = []        if lb.exists():            with open(lb, newline="", encoding="utf-8") as f:                existing = list(csv.DictReader(f))        # drop previous rows for this model so re-runs replace rather than duplicate        existing = [r for r in existing if r.get("model") != model_name]        combined = existing + [{k: ("" if v is None else v) for k, v in r.items()}                               for r in rows_out]        keys = sorted({k for r in combined for k in r},                      key=lambda k: (k != "model", k != "type", k != "n", k))        with open(lb, "w", newline="", encoding="utf-8") as f:            w = csv.DictWriter(f, fieldnames=keys)            w.writeheader()            for r in combined:                w.writerow({k: r.get(k, "") for k in keys})    with open(results_dir / "benchmark_report.txt", "w", encoding="utf-8") as f:        f.write("Ayurveda Shloka Benchmark Report\nModel: %s\n" % model_name)        f.write("=" * 64 + "\n")        for dtype, m in all_metrics.items():            f.write("\n[%s] (n=%s)\n" % (dtype, m.get("n")))            for k, v in m.items():                if k != "n":                    f.write("  %-24s: %s\n" % (k, v))        if confusions:            f.write("\n" + "=" * 64 + "\n")            f.write("Confusion matrices: results/confusion_matrix_<type>_<model>.csv\n")    print("\nDone. Results written to %s/" % results_dir)    return all_metricsdef free_vram():    """Release the previous model's weights - without this the second model in a    batch run OOMs even though it would fit on its own."""    import gc    gc.collect()    try:        torch = _ensure_torch()        if torch.cuda.is_available():            torch.cuda.empty_cache()            torch.cuda.ipc_collect()            print("  VRAM in use: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))    except Exception:        passdef compare_models(model_names: Sequence[str], **kwargs) -> dict:    """Run several models back to back; failures are recorded, not fatal.    VRAM is released between models, and each model's results are written as soon as    it finishes, so a crash on model 4 does not cost you models 1-3.    """    out = {}    for i, name in enumerate(model_names, 1):        print("\n" + "=" * 74)        print("[%d/%d] %s" % (i, len(model_names), name))        print("=" * 74)        try:            out[name] = run_benchmark(name, **kwargs)        except GatedModelError as e:            print("\n%s\n" % e)            out[name] = {"error": "gated", "detail": str(e)}        except Exception as e:            print("\n!! %s FAILED: %s: %s\n" % (name, type(e).__name__, e))            out[name] = {"error": "%s: %s" % (type(e).__name__, e)}        finally:            free_vram()    ok = [k for k, v in out.items() if "error" not in v]    print("\n%d/%d models scored: %s" % (len(ok), len(model_names), ", ".join(ok) or "none"))    for k, v in out.items():        if "error" in v:            print("  FAILED %s -> %s" % (k, v["error"][:120]))    return out# --------------------------------------------------------------------------# Reporting helpers# --------------------------------------------------------------------------def leaderboard(results_dir="results"):    import pandas as pd    return pd.read_csv(Path(results_dir) / "leaderboard.csv")def pivot_view(results_dir="results", values=None):    import pandas as pd    df = leaderboard(results_dir)    values = values or ["accuracy", "f1_macro", "f1_weighted", "format_compliance",                        "bertscore_f1", "tokens_per_sec"]    values = [v for v in values if v in df.columns]    return df.pivot_table(index="model", columns="type", values=values, aggfunc="last")def recommend(results_dir="results", weights=None):    """Rank models by a composite score to shortlist a fine-tuning base.    Default weighting favours reasoning-heavy types and rewards format compliance,    since a model that cannot emit a clean answer letter is painful to evaluate    even if its underlying knowledge is fine.    """    import pandas as pd    df = leaderboard(results_dir)    weights = weights or {"mcq": 0.35, "assertion": 0.25,                          "fill_blanks": 0.20, "match_column": 0.20}    acc = df.pivot_table(index="model", columns="type", values="accuracy", aggfunc="last")    comp = df.pivot_table(index="model", columns="type", values="format_compliance", aggfunc="last")    speed = df.pivot_table(index="model", columns="type", values="tokens_per_sec", aggfunc="last")    out = pd.DataFrame(index=acc.index)    for t in TASK_TYPES:        out["acc_" + t] = acc[t] if t in acc.columns else float("nan")    out["weighted_accuracy"] = sum(        out["acc_" + t].fillna(0) * w for t, w in weights.items() if "acc_" + t in out)    out["mean_format_compliance"] = comp.mean(axis=1) if not comp.empty else float("nan")    out["mean_tokens_per_sec"] = speed.mean(axis=1) if not speed.empty else float("nan")    out["lift_over_random"] = out["weighted_accuracy"] - 0.25    out["composite"] = (0.80 * out["weighted_accuracy"]                        + 0.20 * out["mean_format_compliance"].fillna(0))    return out.sort_values("composite", ascending=False).round(4)if __name__ == "__main__":    validate_data(os.environ.get("AYUR_DATA_DIR", "data"))

## 5. Write the 100 evaluation questionsThe question sets are embedded in this notebook too, so **nothing needs to be attached**.This cell unpacks them into the working directory as four `.jsonl` files.All 100 items come from the AyurShloka-QA `validation.jsonl` split, so they are disjointfrom the ~21k `train.jsonl` questions you will fine-tune on — no leakage.

In [ ]:
import base64, gzip, jsonfrom pathlib import Path_DATA_B64 = ("...")_bundle = json.loads(gzip.decompress(base64.b64decode(_DATA_B64)).decode("utf-8"))for _name, _text in _bundle.items():    Path(_name).write_text(_text, encoding="utf-8")    print("wrote %-28s %3d questions" % (_name, len(_text.strip().splitlines())))

## 6. 🔧 CONFIG — **edit this cell**`MODEL_NAME` is the only thing you must set. Everything else is already in place.Candidates that fit a T4 (≤8B, all public):**Open — no token needed:**```Qwen/Qwen2.5-3B-Instruct           Qwen/Qwen2.5-7B-Instructmicrosoft/Phi-3-mini-4k-instruct   bharatgenai/AyurParambharatgenai/Param-1-2.9B-Instruct  sarvamai/sarvam-1ai4bharat/Airavata                 unsloth/Llama-3.2-3B-Instructunsloth/gemma-2-2b-it              unsloth/gemma-2-9b-it   (needs 4-bit)```**Gated — licence + `HF_TOKEN` required:**```meta-llama/Llama-3.2-3B-Instruct   google/gemma-2-2b-itmeta-llama/Llama-3.1-8B-Instruct   google/gemma-2-9b-it```The `unsloth/*` repos are ungated re-uploads of the same weights, so use those if youwould rather not deal with the licence gate. Section 8 checks access before anythingdownloads.Benchmarking **AyurParam** first gives you the number to beat.

In [ ]:
# ============================ EDIT ME ============================MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"     # <-- the model you want to score# =================================================================import sysfrom pathlib import PathRESULTS_DIR = "results"DATA_DIR = Path(".")             # sections 4 and 5 wrote everything hereif "." not in sys.path:    sys.path.insert(0, ".")print("MODEL_NAME:", MODEL_NAME)missing = [f for f in _bundle if not (DATA_DIR / f).exists()]print("Missing files:", missing or "none — ready to run")

## 7. Import the engine and validate the data`ayur_benchmark` is import-safe — importing it downloads nothing and loads no model.`reload` means any edit you make to `ayur_benchmark.py` takes effect without restartingthe kernel.

In [ ]:
import importlibimport ayur_benchmark as abimportlib.reload(ab)          # so edits to the .py take effect without restartingprint("Task types:", ab.TASK_TYPES)ab.validate_data(DATA_DIR)

## 8. Access check — do this before downloading anythingA gated repo fails only after transformers tries to fetch its config, which is aconfusing place to discover you need a licence. This checks up front, costs one smallHTTP call per model, and names an ungated mirror where one exists.

In [ ]:
CHECK = [    MODEL_NAME,    "Qwen/Qwen2.5-3B-Instruct",    "microsoft/Phi-3-mini-4k-instruct",    "meta-llama/Llama-3.2-3B-Instruct",    "google/gemma-2-2b-it",    "bharatgenai/AyurParam",]USABLE = ab.preflight(sorted(set(CHECK)))

## 9. Inspect a couple of itemsWorth eyeballing once so you know what the model is actually being asked.

In [ ]:
import json, textwrapfor dtype in ab.TASK_TYPES:    rows = ab.load_jsonl(DATA_DIR / ab.FILENAME_MAP[dtype])    r = rows[0]    print("=" * 78)    print("%s  |  %s  |  %s" % (dtype.upper(), r["source"], r["difficulty"]))    print("=" * 78)    print(textwrap.fill(r["instruction"], 76)[:700])    for L, txt in json.loads(r["input"]).items():        print("  (%s) %s" % (L, txt[:110]))    print("  ANSWER: %s -> %s" % (r["output"], r["output_text"][:90]))    print()

## 10. Preview the exact prompt the model will receiveIf a model scores near zero, look here first — it is almost always a prompt/templateproblem, not a knowledge problem.

In [ ]:
from transformers import AutoTokenizer_row = ab.load_jsonl(DATA_DIR / ab.FILENAME_MAP["mcq"])[0]_tok = Nonetry:    _tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=ab.hf_token())except Exception as e:    if ab._is_gated_error(e):        print(ab._gated_message(MODEL_NAME))    else:        print("Tokenizer preview unavailable (%s: %s)" % (type(e).__name__, str(e)[:160]))    print("\nShowing the plain-text prompt instead:\n")print("--- WITH chat template (use for *-Instruct / *-it models) ---")print(ab.build_prompt(_row, "mcq", _tok, use_chat_template=_tok is not None)[:900])print("\n--- WITHOUT chat template (use for base models) ---")print(ab.build_prompt(_row, "mcq", None, use_chat_template=False)[:900])

## 11. Sanity run (fast, writes nothing)`limit=4` per type and `save_results=False`, so this can never pollute `leaderboard.csv`.Confirm **format_compliance is near 1.0** before spending time on the full run.

In [ ]:
ab.run_benchmark(    MODEL_NAME,    data_dir=DATA_DIR,    results_dir=RESULTS_DIR,    device="cuda",    limit=4,    batch_size=4,    skip_bertscore=True,     # skip the 700MB BERTScore download for a smoke test    save_results=False,)

## 12. 🔧 Full run — **tune these per model*** `BATCH_SIZE` — drop to 2 if you hit OOM; raise to 8 for small models.* `LOAD_IN_4BIT` — set `True` for 7-9B models so they fit on one T4.* `USE_CHAT_TEMPLATE` — `True` for instruct/chat models, `False` for base models.* `MAX_NEW_TOKENS` — 24 is plenty; these are letter-answer tasks. Raise to 128 only  if you want to inspect the model's reasoning in the row-level CSV.* `USE_CONTEXT` — `False` tests recall of the classical texts (what you want when  choosing a fine-tuning base). `True` supplies the source passage, turning the task  into reading comprehension.* `TRUST_REMOTE_CODE` — leave at `None`. The loader tries the implementation built into  transformers first, then falls back to the repo's own modeling code, and also retries  with `attn_implementation="eager"` and without safetensors. Set `False` to force  native, `True` to force the repo's code.**If a model fails to load with `KeyError: 'type'`** (Phi-3 does this), that is therepo's bundled `modeling_*.py` reading a config key that transformers has renamed. It isnot an out-of-memory error. `TRUST_REMOTE_CODE = None` already handles it; `False` forcesthe fix.**Real OOM** looks like `torch.cuda.OutOfMemoryError`. Fix by lowering `BATCH_SIZE` to 2,or setting `LOAD_IN_4BIT = True`. A 3B model in fp16 needs ~7 GB and fits a T4 easily atbatch 4 — if a 3B model OOMs, a previous model is still holding VRAM: run`ab.free_vram()` or restart the kernel.

In [ ]:
BATCH_SIZE        = 4LOAD_IN_4BIT      = FalseUSE_CHAT_TEMPLATE = TrueMAX_NEW_TOKENS    = 24USE_CONTEXT       = FalseFORCE_SIZE        = False      # True to allow models above the 8B capTRUST_REMOTE_CODE = None       # None = native first, then repo code. See note below.metrics = ab.run_benchmark(    MODEL_NAME,    data_dir=DATA_DIR,    results_dir=RESULTS_DIR,    device="cuda",    batch_size=BATCH_SIZE,    max_new_tokens=MAX_NEW_TOKENS,    load_in_4bit=LOAD_IN_4BIT,    use_chat_template=USE_CHAT_TEMPLATE,    use_context=USE_CONTEXT,    force_size=FORCE_SIZE,    trust_remote_code=TRUST_REMOTE_CODE,)

## 13. This run's report

In [ ]:
print(open(f"{RESULTS_DIR}/benchmark_report.txt").read())

## 14. Leaderboard — every model scored so farOne row per `(model, task type)`. Re-running a model **replaces** its rows rather thanappending duplicates.

In [ ]:
import pandas as pdpd.set_option("display.max_columns", None)pd.set_option("display.width", 240)lb = ab.leaderboard(RESULTS_DIR)lb

## 15. Pivot — one row per model

In [ ]:
ab.pivot_view(RESULTS_DIR, values=["accuracy", "f1_macro", "format_compliance", "bertscore_f1", "tokens_per_sec"])

## 16. 🔧 Recommendation — which model to fine-tuneComposite = 80% weighted accuracy + 20% format compliance. Adjust `WEIGHTS` if sometask types matter more to you.Read `lift_over_random` first: **0.25 is chance**. A model near 0.25 knows nothinguseful about these texts yet — which is not disqualifying for a fine-tuning base, butit means the benchmark cannot distinguish it from a coin flip.

In [ ]:
WEIGHTS = {"mcq": 0.35, "assertion": 0.25, "fill_blanks": 0.20, "match_column": 0.20}ab.recommend(RESULTS_DIR, weights=WEIGHTS)

## 17. MCQ-style confusion matrixRows = true option letter, columns = predicted. A column piled up on one letter meansthe model has a **position bias** (it always answers "A"). A large `NONE` column meansit never emitted a parseable answer.

In [ ]:
import refrom pathlib import Pathtry:    from IPython.display import displayexcept ImportError:              # running outside a notebook kernel    display = printsafe = re.sub(r"[^A-Za-z0-9_.-]+", "_", MODEL_NAME)for dtype in ab.TASK_TYPES:    p = Path(RESULTS_DIR) / f"confusion_matrix_{dtype}_{safe}.csv"    if p.exists():        print(f"\n=== {dtype} ===")        display(pd.read_csv(p, index_col=0))    else:        print(f"(no confusion matrix for {dtype})")

## 18. Error analysis — where is it losing points?Accuracy broken down by source book, question subtype and difficulty. This is whattells you *what* to over-sample when you build the fine-tuning mix.

In [ ]:
rows = pd.read_csv(f"{RESULTS_DIR}/benchmark_results_{safe}.csv")print("--- accuracy by task type ---")display(rows.groupby("type")["correct"].agg(["mean", "count"]).round(3))print("--- accuracy by question subtype ---")display(rows.groupby("question_subtype")["correct"].agg(["mean", "count"]).round(3)            .sort_values("mean"))print("--- accuracy by source book ---")display(rows.groupby("source_book")["correct"].agg(["mean", "count"]).round(3)            .sort_values("mean"))print("--- accuracy by difficulty ---")display(rows.groupby("difficulty")["correct"].agg(["mean", "count"]).round(3))

## 19. Row-level predictions vs. gold

In [ ]:
cols = ["type", "id", "source", "gold_letter", "pred_letter", "correct", "pred"]display(rows[cols].head(30))print("\n--- items it got WRONG (first 10) ---")for _, r in rows[rows["correct"] == 0].head(10).iterrows():    print("\n[%s] %s" % (r["type"], r["source"]))    print("  gold: %s   pred: %s" % (r["gold_letter"], r["pred_letter"]))    print("  raw : %s" % str(r["pred"])[:200].replace("\n", " ⏎ "))

## 20. Compare several models in one goUncomment and run to queue a batch. Each model is loaded, scored and freed in turn;a failure is recorded and does not abort the rest.

In [ ]:
CANDIDATES = [    "Qwen/Qwen2.5-3B-Instruct",    "microsoft/Phi-3-mini-4k-instruct",    "google/gemma-2-2b-it",    "meta-llama/Llama-3.2-3B-Instruct",   # gated: needs HF_TOKEN    "bharatgenai/AyurParam",              # the Ayurveda baseline]results = ab.compare_models(    CANDIDATES,    data_dir=DATA_DIR, results_dir=RESULTS_DIR, device="cuda",    batch_size=4, max_new_tokens=24, use_chat_template=True,)display(ab.recommend(RESULTS_DIR))

## 21. Download the resultsTwo kinds of zip: one with everything, and one per model so you can hand a singlemodel's numbers over without the rest.**`/kaggle/working/` is wiped when the session ends** — download before you close, orthe leaderboard starts empty next time.

In [ ]:
import shutil, zipfile, refrom pathlib import PathRES = Path(RESULTS_DIR)shutil.make_archive("ayur_benchmark_results_ALL", "zip", RES)   # everythingmodels = set()if (RES / "leaderboard.csv").exists():    import pandas as pd    models = set(pd.read_csv(RES / "leaderboard.csv")["model"])for m in sorted(models):                                        # one zip per model    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", m)    files = [p for p in RES.iterdir() if safe in p.name]    if not files:        continue    with zipfile.ZipFile("ayur_results_%s.zip" % safe, "w", zipfile.ZIP_DEFLATED) as z:        for p in files:            z.write(p, p.name)        z.write(RES / "leaderboard.csv", "leaderboard.csv")print("Zips in /kaggle/working/ - see the Output panel on the right:")for f in sorted(Path(".").glob("*.zip")):    print("   %-54s %7.1f KB" % (f.name, f.stat().st_size / 1024))print()print("Models scored:", ", ".join(sorted(models)) or "none yet")